# IU X-ray dataset — exploratory data analysis

**Dataset:** Indiana University Chest X-ray Collection  
**File:** `annotation.json` — splits: `train`, `val`, `test`  
**Schema per entry:** `id`, `image_path` (list of PNGs), `report` (free text), `split`

This notebook covers:
1. Load & flatten annotation.json into a DataFrame
2. Split sizes
3. Images per patient
4. Report text length distribution
5. Keyword frequency analysis (Impression-style terms)
6. Sample image preview
7. Summary & next steps


## 1. Imports and configuration

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")
%matplotlib inline

DATA_ROOT = Path("/data/SwimMambaT5/iu_xray")
ANNOTATION = DATA_ROOT / "annotation.json"


## 2. Load annotation.json and flatten to DataFrame

In [ ]:
with open(ANNOTATION) as f:
    raw = json.load(f)

print("Splits found:", list(raw.keys()))
print("Counts:", {k: len(v) for k, v in raw.items()})

# Flatten all splits into one DataFrame
records = []
for split, entries in raw.items():
    for entry in entries:
        records.append({
            "id":          entry["id"],
            "split":       entry["split"],
            "n_images":    len(entry["image_path"]),
            "image_paths": entry["image_path"],
            "report":      entry.get("report", ""),
        })

df = pd.DataFrame(records)
print(f"\nTotal entries: {len(df):,}")
df.head()


## 3. Split sizes

In [ ]:
split_counts = df["split"].value_counts().reindex(["train", "val", "test"])
print(split_counts.to_string())

fig, ax = plt.subplots(figsize=(5, 3.5))
split_counts.plot(kind="bar", ax=ax, color=["#0F6E56", "#378ADD", "#D85A30"], rot=0)
ax.set_title("Entries per split")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}",
                (p.get_x() + p.get_width() / 2, p.get_height() + 10),
                ha="center", fontsize=11)
fig.tight_layout()
plt.show()


## 4. Images per patient

Most IU X-ray patients have 2 images (frontal + lateral).
Splits should be made at patient (`id`) level to avoid leakage.

In [ ]:
img_counts = df["n_images"].value_counts().sort_index()
print(img_counts.to_string())

fig, ax = plt.subplots(figsize=(4, 3.5))
img_counts.plot(kind="bar", ax=ax, color="#0F6E56", rot=0)
ax.set_title("Images per patient")
ax.set_xlabel("Number of images")
ax.set_ylabel("Number of patients")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}",
                (p.get_x() + p.get_width() / 2, p.get_height() + 3),
                ha="center", fontsize=11)
fig.tight_layout()
plt.show()


## 5. Report completeness

In [ ]:
missing = df["report"].isna() | (df["report"].str.strip() == "")
print(f"Reports present : {(~missing).sum():,}  ({100*(~missing).mean():.1f}%)")
print(f"Reports missing : {missing.sum():,}  ({100*missing.mean():.1f}%)")


## 6. Report text length distribution

In [ ]:
df["word_count"] = df["report"].dropna().str.split().apply(len)
df["char_count"] = df["report"].dropna().str.len()

print(df["word_count"].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(df["word_count"].dropna(), bins=40, ax=axes[0], color="#1D9E75")
axes[0].set_title("Report length (words)")
axes[0].set_xlabel("Word count")

sns.histplot(df["char_count"].dropna(), bins=40, ax=axes[1], color="#378ADD")
axes[1].set_title("Report length (characters)")
axes[1].set_xlabel("Character count")

fig.tight_layout()
plt.show()


## 7. Report length by split

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for split, color in zip(["train", "val", "test"], ["#0F6E56", "#378ADD", "#D85A30"]):
    subset = df[df["split"] == split]["word_count"].dropna()
    sns.kdeplot(subset, ax=ax, label=split, color=color, fill=True, alpha=0.25)
ax.set_title("Report word count distribution by split")
ax.set_xlabel("Word count")
ax.legend()
fig.tight_layout()
plt.show()


## 8. Keyword frequency in report text

Simple word-match frequency over the full report text.
IU X-ray has no multi-label ground truth — these counts give a rough sense
of class imbalance before applying a proper rule-based labeler.

In [ ]:
KEYWORDS = [
    "cardiomegaly", "opacity", "opacities", "effusion", "edema",
    "pneumonia", "atelectasis", "pneumothorax", "consolidation",
    "nodule", "mass", "fracture", "emphysema", "fibrosis",
    "hernia", "calcified", "degenerative", "normal",
    "no acute", "unremarkable",
]

text_corpus = df["report"].dropna().str.lower()

kw_counts = {}
for kw in KEYWORDS:
    pattern = re.compile(rf"\b{re.escape(kw)}\b")
    kw_counts[kw] = int(text_corpus.str.contains(pattern).sum())

kw_series = pd.Series(kw_counts).sort_values(ascending=False)
print(kw_series.to_string())

fig, ax = plt.subplots(figsize=(7, 6))
kw_series.sort_values().plot(kind="barh", ax=ax, color="#D85A30")
ax.set_title("Keyword mentions across all reports")
ax.set_xlabel("Number of reports containing keyword")
fig.tight_layout()
plt.show()


## 9. Most frequent content words in reports

In [ ]:
STOPWORDS = {
    "the", "and", "is", "are", "in", "of", "no", "a", "an", "to",
    "within", "normal", "limits", "there", "or", "with", "at", "on",
    "seen", "noted", "present", "appears", "appear", "appears", "for",
    "from", "this", "not", "it", "by", "as", "be", "has", "have",
}

all_words = (
    df["report"].dropna().str.lower()
    .str.replace(r"[^a-z\s]", " ", regex=True)
    .str.split()
    .explode()
)
word_freq = (
    all_words[~all_words.isin(STOPWORDS)]
    .value_counts()
    .head(25)
)

fig, ax = plt.subplots(figsize=(8, 5))
word_freq.sort_values().plot(kind="barh", ax=ax, color="#1B2A4A")
ax.set_title("Top 25 content words in reports (stopwords removed)")
ax.set_xlabel("Count")
fig.tight_layout()
plt.show()


## 10. Sample image preview

Shows the first image for a few patients across splits.

In [ ]:
def show_sample_images(df, data_root, n=4, split="train"):
    subset = df[df["split"] == split].head(n)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    for ax, (_, row) in zip(axes, subset.iterrows()):
        img_path = data_root / row["image_paths"][0]
        if img_path.exists():
            ax.imshow(Image.open(img_path), cmap="gray")
            ax.set_title(f"{row['id']}\n({row['n_images']} images)", fontsize=8)
        else:
            ax.text(0.5, 0.5, f"not found:\n{img_path.name}",
                    ha="center", va="center", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"Sample images — {split} split", fontsize=13)
    fig.tight_layout()
    plt.show()

show_sample_images(df, DATA_ROOT, n=4, split="train")


## 11. Summary and next steps

### Key findings
- **Schema:** each entry has `id`, 1-2 `image_path`s, and a single free-text `report`
- **No multi-label ground truth** — labels must be derived from report text via a rule-based labeler (e.g. CheXpert-style)
- **Most patients have 2 images** (frontal + lateral) — train/val/test splits must be done at patient (`id`) level to avoid leakage
- **Report length** is moderate (~30–60 words typically) — suitable for a clinical text encoder (ClinicalBERT / BioBERT)
- **Dominant findings** in text: cardiomegaly, effusion, opacity, hernia — class imbalance is significant

### Next steps
1. Apply rule-based labeler to `report` text to produce multi-label targets
2. Build patient-level DataLoader (image + report → label)
3. Train vision-only baseline (ResNet/ViT on images)
4. Train text-only baseline (ClinicalBERT on reports)
5. Design and train multimodal fusion model
